# 01 — Image Extraction and Filtering

Extract embedded PDF images, apply the original filtering rules, and save compatible manifests.
Production logic lives in `intelligent_scada.ingestion`. Keep manual review in this notebook.

In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "intelligent_scada").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Intelligent SCADA project root.")

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from intelligent_scada.config import Settings

settings = Settings.from_env(project_root=project_root, env_override=True).for_default_document()
settings.ensure_output_dirs()


In [ ]:
from intelligent_scada.ingestion.image_extractor import extract_images
from intelligent_scada.ingestion.image_filter import filter_images

manifest = extract_images(
    settings.require_default_pdf(),
    settings.raw_images_dir,
    settings.image_manifest_path,
    output_format=settings.output_image_format,
    filename_template=settings.image_filename_template,
)

filtered = filter_images(
    manifest,
    settings.raw_images_dir,
    settings.filtered_keep_dir,
    settings.filtered_discard_dir,
    settings.filter_manifest_path,
    settings,
)

print(f"Extracted: {len(manifest)}")
print(f"Keep: {sum(item['decision'] == 'keep' for item in filtered)}")
print(f"Discard: {sum(item['decision'] == 'discard' for item in filtered)}")
print(f"Manifest: {settings.filter_manifest_path}")


## Optional manual review

Move files between the keep/discard folders, then run the next cell. Manual decisions are
recorded with `source="manual_review"`.

In [ ]:
from intelligent_scada.ingestion.image_filter import rebuild_manual_review_manifest

APPLY_MANUAL_REVIEW = False

if APPLY_MANUAL_REVIEW:
    reviewed = rebuild_manual_review_manifest(
        manifest,
        settings.filtered_keep_dir,
        settings.filtered_discard_dir,
        settings.filter_manifest_path,
    )
    print(f"Reviewed records: {len(reviewed)}")
else:
    print("Manual review not applied. Set APPLY_MANUAL_REVIEW = True after moving files.")
